# วิเคราะห์ความสัมพันธ์ของข้อมูลต้นทาง

Notebook นี้เปรียบเทียบ NSL-KDD กับ UNSW-NB15 จาก train_full เพื่อช่วยตรวจความแตกต่างของ feature ก่อนเลือกใช้ร่วมกัน เป็นการวิเคราะห์เชิงพรรณนา ไม่ได้ฝึกโมเดล

แถวจากสอง dataset ไม่ใช่เหตุการณ์เดียวกัน จึงคำนวณ correlation ภายในแต่ละชุดแล้วค่อยเปรียบเทียบ ไม่จับคู่แถวข้ามแหล่ง


## 1. อ่าน train_full และตรวจตาราง

labels ใช้แบ่งสรุป Normal/Attack เท่านั้น; ไม่ใช้ validation หรือ test


In [ ]:
from pathlib import Path
from itertools import combinations
import json
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, chi2_contingency
from scipy.spatial.distance import jensenshannon


def find_central_dir(start=None):
    origin = Path(start or Path.cwd()).resolve()
    for candidate in (origin, *origin.parents):
        if candidate.name == "CENTRAL" and (candidate / "build_dataset.ipynb").is_file():
            return candidate
        nested = candidate / "data-pipeline" / "CENTRAL"
        if (nested / "build_dataset.ipynb").is_file():
            return nested
    raise FileNotFoundError(f"หา CENTRAL จากตำแหน่งนี้ไม่พบ: {origin}")


CENTRAL_DIR = find_central_dir()
INPUT_DIR = CENTRAL_DIR / "output" / "splits" / "train_full"
OUTPUT_DIR = CENTRAL_DIR / "output" / "reports" / "dataset_relationships"
NUMERIC_FEATURES = ["duration", "bytes_forward", "bytes_backward"]
CATEGORICAL_FEATURES = ["protocol", "service_group", "conn_state_group"]
SOURCES = ["NSL-KDD", "UNSW-NB15"]

features, metadata, labels = [
    pd.read_csv(INPUT_DIR / f"{name}.csv", low_memory=False)
    for name in ("features", "metadata", "labels")
]
for table in (features, metadata, labels):
    if not table["record_id"].is_unique:
        raise ValueError("พบ record_id ซ้ำใน train_full")
if not set(features.record_id) == set(metadata.record_id) == set(labels.record_id):
    raise ValueError("record_id ของ features, metadata และ labels ไม่ตรงกัน")

data = (
    features.merge(metadata[["record_id", "source_dataset"]], on="record_id", validate="one_to_one")
    .merge(labels[["record_id", "label_binary"]], on="record_id", validate="one_to_one")
)
if not np.isfinite(data[NUMERIC_FEATURES].to_numpy()).all() or (data[NUMERIC_FEATURES] < 0).any().any():
    raise ValueError("numeric feature มีค่า missing, infinite หรือค่าติดลบ")
if not data[CATEGORICAL_FEATURES].notna().all().all():
    raise ValueError("categorical feature มี missing value")
missing_sources = set(SOURCES).difference(data["source_dataset"].unique())
if missing_sources:
    raise ValueError(f"ไม่พบ source ที่ต้องการเปรียบเทียบ: {sorted(missing_sources)}")
print("Input:", INPUT_DIR)
print("Records:", f"{len(data):,}")

## 2. คำนวณ correlation และ distribution shift

KS และ Jensen-Shannon อธิบายความต่างของการแจกแจง ไม่ได้ยืนยันว่าความหมายของ feature เท่ากัน


In [ ]:
counts, correlations, shifts, categories, associations = [], [], [], [], []

for scope, subset in [("all", data), ("normal", data[data.label_binary == 0]), ("attack", data[data.label_binary == 1])]:
    groups = {source: subset[subset.source_dataset == source] for source in SOURCES}
    for source, group in groups.items():
        counts.append({"scope": scope, "source": source, "rows": len(group)})
        for transform, values in [("raw", group[NUMERIC_FEATURES]), ("log1p", np.log1p(group[NUMERIC_FEATURES]))]:
            for method in ("pearson", "spearman"):
                matrix = values.corr(method=method)
                for first, second in combinations(NUMERIC_FEATURES, 2):
                    correlations.append({
                        "scope": scope, "source": source, "transform": transform, "method": method,
                        "feature_a": first, "feature_b": second,
                        "correlation": matrix.loc[first, second],
                    })
        for first, second in combinations(CATEGORICAL_FEATURES, 2):
            table = pd.crosstab(group[first], group[second])
            dimension = min(table.shape) - 1
            value = np.sqrt(chi2_contingency(table, correction=False)[0] / table.to_numpy().sum() / dimension) if dimension > 0 else np.nan
            associations.append({"scope": scope, "source": source, "feature_a": first, "feature_b": second, "cramers_v": value})

    for feature in NUMERIC_FEATURES:
        first = groups[SOURCES[0]][feature].to_numpy()
        second = groups[SOURCES[1]][feature].to_numpy()
        shifts.append({
            "scope": scope, "feature": feature, "ks_statistic": ks_2samp(first, second, method="asymp").statistic,
            "nsl_median": np.median(first), "unsw_median": np.median(second),
            "nsl_q95": np.quantile(first, .95), "unsw_q95": np.quantile(second, .95),
            "nsl_zero_rate": np.mean(first == 0), "unsw_zero_rate": np.mean(second == 0),
        })
    for feature in CATEGORICAL_FEATURES:
        values = sorted(subset[feature].unique())
        proportions = [groups[source][feature].value_counts(normalize=True).reindex(values, fill_value=0) for source in SOURCES]
        distance = jensenshannon(proportions[0], proportions[1], base=2)
        for value in values:
            categories.append({
                "scope": scope, "feature": feature, "category": value,
                "nsl_proportion": proportions[0][value], "unsw_proportion": proportions[1][value],
                "js_distance": distance,
            })

## 3. บันทึกผล

ไฟล์ CSV เก็บผลราย feature และ manifest บอกข้อจำกัดของการตีความ


In [ ]:
outputs = {
    "sample_counts": pd.DataFrame(counts),
    "numeric_correlations": pd.DataFrame(correlations),
    "numeric_distribution_shift": pd.DataFrame(shifts),
    "categorical_distribution_shift": pd.DataFrame(categories),
    "categorical_associations": pd.DataFrame(associations),
}
correlation_table = outputs["numeric_correlations"]
comparison = correlation_table.pivot(
    index=["scope", "transform", "method", "feature_a", "feature_b"],
    columns="source", values="correlation",
).reset_index()
comparison["absolute_difference"] = (comparison[SOURCES[0]] - comparison[SOURCES[1]]).abs()
outputs["correlation_comparison"] = comparison

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, table in outputs.items():
    table.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)

manifest = {
    "input_split": "train_full",
    "test_used": False,
    "label_used_for": "descriptive normal/attack stratification only",
    "unpaired_datasets": True,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "notes": [
        "Correlation compares columns within each dataset; rows are not paired across sources.",
        "KS and Jensen-Shannon are descriptive distances, not semantic equivalence tests.",
        "Normal/attack comparisons do not remove all protocol/service composition differences.",
        "Cramers V is uncorrected and not directional.",
        "Byte semantics still require checking source documentation.",
    ],
}
(OUTPUT_DIR / "analysis_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(outputs["sample_counts"].to_string(index=False))
print()
print("Analysis outputs:", OUTPUT_DIR)